<a href="https://colab.research.google.com/github/Camiau20/Camiau20/blob/main/notebooks/09_TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Entrenamiento, Predicción y Evaluación de un modelo LSTM

#### **Importación de Datos**

In [1]:
!pip install pytorch-forecasting==1.4.0 pytorch-lightning==2.4.0 torchmetrics==1.3.1 ucimlrepo
import requests
import pandas as pd
import numpy as np
import sys, os, math, typing as t
import warnings
import time

import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
import pytorch_lightning as pl

from datetime import timedelta
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_lightning import LightningModule

from io import BytesIO
from dataclasses import dataclass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.9/260.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.4/840.4 kB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 52.7 MB/s eta 0:00:00


In [3]:
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True
})

warnings.filterwarnings('ignore')



In [4]:
# Importar datos
DATA_GITHUB_URL = 'https://raw.githubusercontent.com/DCajiao/Time-series-forecast-of-energy-consumption-in-Tetouan-City/refs/heads/main/data/enriched_zone1_power_consumption_of_tetouan_city.csv'

# Descargar los datos desde github
response = requests.get(DATA_GITHUB_URL)

# Convertir en un df desde el xlsx de github
df = pd.read_csv(BytesIO(response.content), sep=',')

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

# df = df.set_index("datetime") Esto ya no se hace

t0 = df["datetime"].min()
df["time_idx"] = ((df["datetime"] - t0) / pd.Timedelta(minutes=10)).astype(int)

# 4. Validación de columnas mínimas
expected_cols = {"temperature","humidity","general_diffuse_flows","zone_1"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas en el dataset: {missing}. "
                     f"Columnas disponibles: {df.columns.tolist()}")

# 5. Features de calendario
df["day_of_week"] = df["datetime"].dt.dayofweek

print(df.head())

             datetime  temperature  humidity  wind_speed  \
0 2017-01-01 00:00:00        6.559      73.8       0.083   
1 2017-01-01 00:10:00        6.414      74.5       0.083   
2 2017-01-01 00:20:00        6.313      74.5       0.080   
3 2017-01-01 00:30:00        6.121      75.0       0.083   
4 2017-01-01 00:40:00        5.921      75.7       0.081   

   general_diffuse_flows       zone_1  is_weekend  is_holiday  hour  day  \
0                  0.051  34055.69620        True        True     0    1   
1                  0.070  29814.68354        True        True     0    1   
2                  0.062  29128.10127        True        True     0    1   
3                  0.091  28228.86076        True        True     0    1   
4                  0.048  27335.69620        True        True     0    1   

   month  time_idx  day_of_week  
0      1         0            6  
1      1         1            6  
2      1         2            6  
3      1         3            6  
4      1    

#### **Definición de Funciones y Partición de los datos**

In [5]:
prediction_length = 24 * 6        # 24 horas * 6 (10-min steps) = 144
max_encoder_length = 7 * 24 * 6   # 7 días históricos (~1008) — ajustar si memoria se agota

# crear columna "group", es constante, porque solo tenemos zone_1
df['zone'] = 'zone_1'

training_cutoff = df['time_idx'].max() - prediction_length

training = TimeSeriesDataSet(
    df[df.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="zone_1",
    group_ids=["zone"], #No tenemos esto realmente, todas son "zone_1"

    min_encoder_length=max_encoder_length//2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=prediction_length,

    static_categoricals=["zone"], #No estoy segura de esto, si de pronto integrar las otras zonas-
    time_varying_known_reals=["time_idx", "hour", "day", "day_of_week", "month", "is_weekend", "is_holiday", "temperature", "humidity", "wind_speed", "general_diffuse_flows"],
    time_varying_unknown_reals=["zone_1"],

    target_normalizer=GroupNormalizer(groups=["zone"]),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# validación (reutiliza normalizadores del training)
validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

train_dataloader = training.to_dataloader(train=True, batch_size=64, num_workers=4)
val_dataloader = validation.to_dataloader(train=False, batch_size=64*4, num_workers=4)

#### **Entrenamiento**

In [6]:
class TFTLightningWrapper(LightningModule):
    def __init__(self, tft):
        super().__init__()
        self.tft = tft
        self.loss = tft.loss
        self.save_hyperparameters(ignore=["tft","loss"])

    def forward(self, x):
        return self.tft(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        out = self(x)
        loss = self.loss(out[0], y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        out = self(x)
        loss = self.loss(out[0], y)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return self.tft.configure_optimizers()


In [7]:
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

early_stop = EarlyStopping(monitor="val_loss", patience=5, mode="min")
checkpoint = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1, filename="tft-{epoch:02d}-{val_loss:.4f}")
lr_monitor = LearningRateMonitor(logging_interval='epoch')

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1 if torch.cuda.is_available() else None,
    callbacks=[early_stop, checkpoint, lr_monitor],
    gradient_clip_val=0.1,
    log_every_n_steps=10
)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=32,
    loss=QuantileLoss(),
)

tft_module = TFTLightningWrapper(tft)
trainer.fit(tft_module, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
# el mejor checkpoint queda guardado en checkpoint.best_model_path


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name | Type                      | Params | Mode 
-----------------------------------------------------------
0 | tft  | TemporalFusionTransformer | 408 K  | train
1 | loss | QuantileLoss              | 0      | train
-----------------------------------------------------------
408 K     Trainable params
0         Non-trainable params
408 K     Total params
1.633     Total estimated model params size (MB)
575       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

RuntimeError: Expected all tensors to be on the same device, but got tensors is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_cat)

---

#### **Predicción**

In [ ]:
best_model_path = trainer.checkpoint_callback.best_model_path
tft_loaded = TFTLightningWrapper.load_from_checkpoint(best_model_path, tft=tft, loss=QuantileLoss())
tft = tft_loaded.tft  # modelo real

# predicción raw (necesaria para interpretar internamente)
raw_predictions = tft.predict(val_dataloader, mode="raw", return_x=True)

# determinar índice de la mediana de forma dinámica (evita hardcode)
quantiles = list(tft.loss.quantiles)    # ej. [0.02,0.1,0.25,0.5,...]
median_idx = quantiles.index(0.5) if 0.5 in quantiles else len(quantiles)//2

# extraer mediana de predicción (shape: n_samples, prediction_length)
y_pred = raw_predictions.output[0][:, :, median_idx].cpu().numpy()

# Construir y_true (agregando batches)
y_true_list = []
for x_batch, y_batch in val_dataloader:
    # y_batch[0] contiene target según la estructura típica de pytorch-forecasting
    y_true_list.append(y_batch[0].cpu().numpy())

y_true = np.concatenate(y_true_list, axis=0)  # shape: (n_samples, prediction_length, 1)
y_true = y_true.squeeze(-1)  # (n_samples, prediction_length)

# aplanar para métricas comparables
y_true_flat = y_true.flatten()
y_pred_flat = y_pred.flatten()

# métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(y_true_flat, y_pred_flat)
rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
non_zero_mask = y_true_flat != 0
mape = np.mean(np.abs((y_true_flat[non_zero_mask] - y_pred_flat[non_zero_mask]) / y_true_flat[non_zero_mask]))*100

print(f"MAE: {mae:.3f} | RMSE: {rmse:.3f} | MAPE: {mape:.2f}%")

## INTERPRETABILIDAD:

In [ ]:
interpretation = tft.interpret_output(raw_predictions.output, reduction="sum")
figs = tft.plot_interpretation(interpretation)

# Para inspección programática: ver las claves y shapes
for k, v in interpretation.items():
    print(k, getattr(v, "shape", None))